# TV3 - LSTM và Conv1D cho SMS spam
Chạy theo thứ tự. Notebook là hướng dẫn chạy lại trên Colab, chưa được thực thi trong Colab ở lần bàn giao này.
Kết quả trong gói đã chạy thực tế trên Windows CPU.
Chuẩn bị `Bo_sung_TV1_TV2.zip` và `Bo_sung_TV3.zip` đã nhận. Giữ hai file ZIP đó trong máy để upload ở bước 2.


## 1 Tải repo gốc

In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/NIC_AI6_PROJECT')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/mochilemonmochi890-cmd/NIC_AI6_PROJECT.git', str(repo)], check=True)
%cd /content/NIC_AI6_PROJECT


## 2 Upload hai gói bổ sung và giải nén vào repo

In [ ]:
from google.colab import files
import zipfile
uploaded = files.upload()
for filename in uploaded:
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename) as archive:
            for entry in archive.infolist():
                target = (repo / entry.filename).resolve()
                if not target.is_relative_to(repo.resolve()):
                    raise ValueError('Đường dẫn ZIP không hợp lệ')
            archive.extractall(repo)
assert (repo / 'traditional_ml/bo_sung_tv2/vi/data/train.csv').exists(), 'Cần gói TV1/TV2'
assert (repo / 'deep_learning/bo_sung_tv3/train_dl.py').exists(), 'Cần gói TV3'


## 3 Cài thư viện và chạy huấn luyện
Huấn luyện sẽ ghi lại đầu ra TV3. Muốn xem kết quả có sẵn thì bỏ qua ô huấn luyện.

In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'deep_learning/bo_sung_tv3/requirements.txt'], check=True)


In [ ]:
subprocess.run([sys.executable, 'deep_learning/bo_sung_tv3/train_dl.py', '--language', 'both'], check=True)
subprocess.run([sys.executable, 'deep_learning/bo_sung_tv3/compare_templates.py'], check=True)


## 4 Xem kết quả và biểu đồ

In [ ]:
import pandas as pd
from IPython.display import Image, display
for language in ['vi', 'uci']:
    base = repo / 'deep_learning/bo_sung_tv3' / language
    print(language)
    display(pd.read_csv(base / 'results/comparison_ml_dl.csv'))
    display(Image(filename=str(base / 'figures/comparison_ml_dl.png'), width=640))


## 5 Thử dự đoán

In [ ]:
subprocess.run([sys.executable, 'deep_learning/bo_sung_tv3/predict_dl.py', '--language', 'vi', '--text', 'Ma OTP la 123456'], check=True)


## 6 Kiểm tra model đã lưu và tải phần TV3

In [ ]:
subprocess.run([sys.executable, 'deep_learning/bo_sung_tv3/check_handoff.py'], check=True)
output = Path('/content/TV3_ket_qua.zip')
base = repo / 'deep_learning/bo_sung_tv3'
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as archive:
    for item in base.rglob('*'):
        if item.is_file() and not any(part in ['.venv', '__pycache__'] for part in item.parts):
            archive.write(item, item.relative_to(repo))
files.download(str(output))
